# Cross-model mechanistic bias study (Kaggle)

Runs GPT-2 small, GPT-2 Large, and Pythia-1B on the original 200-pair gender thesis dataset. It creates occupation-grouped train/validation/test splits, fits separate directions per model, selects intervention settings on validation only, evaluates locked settings on held-out occupations, reports both the token-aware and previous-thesis L2 metrics, and writes PNG/PDF figures with occupation-cluster confidence intervals.

**Settings:** GPU enabled (one T4 is sufficient) and Internet on. Pythia-1B is public and requires no Hugging Face token.

**Scope:** The original dataset supports gender/occupation analysis. It does not provide a broad, independently annotated dynamic-gate evaluation set or a demographic evaluation set. Report those as limitations; do not claim broad gate validity from the matched controls alone. CoLA perplexity is a language-modeling measure, not CoLA task accuracy.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO=Path('/kaggle/working/MI_Bias-Cross-Model-Extension')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/aniruddhak94/MI_Bias-Cross-Model-Extension.git',str(REPO)],check=True)
else: subprocess.run(['git','pull','--ff-only','origin','main'],cwd=REPO,check=True)
os.chdir(REPO)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','graphviz','libgraphviz-dev','pkg-config'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
import transformers, transformer_lens
print('Transformers:',transformers.__version__,'| TransformerLens:',transformer_lens.__version__)
if transformers.__version__ != '4.57.6': raise RuntimeError(f'Expected transformers==4.57.6, got {transformers.__version__}; rerun Cell 1 after checking requirements.txt')
print('Ready:',REPO)

In [ ]:
import torch, shutil
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); print(f'GPU {i}: {p.name}, {p.total_memory/2**30:.2f} GiB')
print('Free disk GiB:',round(shutil.disk_usage('/kaggle/working').free/2**30,2))
if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU in Kaggle settings before continuing')
print('No gated-model token is required for this model set.')

## Prepare the thesis prompt data

This cell uses the original 200-pair thesis gender dataset included in the repository. It keeps every wording variant of an occupation in one split (about 60/20/20) and creates the matched gate labels. The test split is about 40 prompts; report its group-level uncertainty and avoid treating the earlier full-200 run as held-out evidence.

In [ ]:
from pathlib import Path
import subprocess, sys
DATA=REPO/'kaggle_data'
subprocess.run([sys.executable,'scripts/prepare_thesis_dataset.py','Dataset/thesis_gender_bias_200.json','--output-dir',str(DATA),'--seed','42'],cwd=REPO,check=True)
subprocess.run([sys.executable,'scripts/validate_splits.py','--train',str(DATA/'train.json'),'--validation',str(DATA/'validation.json'),'--test',str(DATA/'test.json')],cwd=REPO,check=True)
print('Using original thesis prompts; inspect dataset_manifest.json before the model run.')

Optional capability report: CoLA validation perplexity is scored after intervention settings are locked. It is reported only as language-modeling perplexity, not task accuracy.

In [ ]:
from datasets import load_dataset
cola=load_dataset('nyu-mll/glue','cola',split='validation')
cola_rows=[{'id':str(i),'text':r['sentence']} for i,r in enumerate(cola) if r.get('sentence','').strip()]
(DATA/'cola_validation_text.json').write_text(json.dumps(cola_rows,indent=2),encoding='utf-8')
print('CoLA validation sentences:',len(cola_rows))

## Run the cross-model pipeline

The script saves results by run ID and skips completed outputs. If the session stops, rerun with the same run ID to resume. Validation selection uses the largest mean relative bias reduction; review this rule and adjust if your thesis protocol sets a capability constraint.

In [ ]:
RID_FILE=Path('/kaggle/working/mi_bias_thesis_gender_run_id.txt')
if RID_FILE.exists(): RUN_ID=RID_FILE.read_text().strip()
else:
    from datetime import datetime,timezone
    RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); RID_FILE.write_text(RUN_ID)
subprocess.run([sys.executable,'scripts/run_cross_model_study.py','--data-dir',str(DATA),'--output-root','/kaggle/working/cross_model_runs','--run-id',RUN_ID],cwd=REPO,check=True)
RUN_ROOT=Path('/kaggle/working/cross_model_runs')/RUN_ID
print('Completed run:',RUN_ID,'at',RUN_ROOT)
print('Manifest:',(DATA/'dataset_manifest.json').read_text())

## View figures and package outputs

In [ ]:
from IPython.display import Image,display
for p in sorted((RUN_ROOT/'figures').rglob('*.png')): print(p.relative_to(RUN_ROOT/'figures')); display(Image(filename=str(p)))
shutil.copytree(DATA,RUN_ROOT/'data',dirs_exist_ok=True)
zip_path=shutil.make_archive(f'/kaggle/working/mi_bias_cross_model_{RUN_ID}','zip',root_dir=RUN_ROOT)
print('Download results:',zip_path)
print('Planning estimates: GPT-2 small ~1-3h; GPT-2 Large ~3-8h; Pythia-1B ~1-3h. Actual time depends on Kaggle allocation, downloads, and model load speed.')